# EmoteFlow - CNN Emotion Recognition Training

Train on FER2013, fine-tune on CK+, export to ONNX, and upload to Hugging Face Hub.

**Kaggle Setup:**
1. Add dataset **msambare/fer2013** (image folder version) via `+ Add Input`
2. Optionally add a CK+ dataset for fine-tuning
3. Enable **GPU** under Settings → Accelerator
4. Add your **HF_TOKEN** as a Kaggle Secret (Add-ons → Secrets)
5. Run all cells

## Cell 1: Install Dependencies

In [ ]:
!pip install -q tf2onnx huggingface_hub onnxruntime

## Cell 2: Load Kaggle Secrets & Imports

In [ ]:
import os

# Load HF_TOKEN from Kaggle Secrets
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN loaded from Kaggle Secrets")
except Exception:
    print("Warning: Could not load HF_TOKEN. Hugging Face upload will be skipped.")

import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks, optimizers
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight
import matplotlib.pyplot as plt
import seaborn as sns

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {len(tf.config.list_physical_devices('GPU')) > 0}")

## Cell 3: Configuration

In [ ]:
# ─── Configuration ───────────────────────────────────────────────────────────

IMG_SIZE = 48
NUM_CLASSES = 7
BATCH_SIZE = 64
EPOCHS_INITIAL = 50
EPOCHS_FINETUNE = 20
LEARNING_RATE = 1e-3
FINETUNE_LR = 1e-4

EMOTION_LABELS = ["Angry", "Disgust", "Fear", "Happy", "Sad", "Surprise", "Neutral"]

# Kaggle paths (adjust if your dataset slug differs)
FER2013_TRAIN_DIR = "/kaggle/input/fer-2013/train"
FER2013_TEST_DIR = "/kaggle/input/fer-2013/test"
CK_PLUS_DIR = "/kaggle/input/ck-plus"  # optional fine-tuning dataset

OUTPUT_DIR = "/kaggle/working"
MODEL_SAVE_PATH = os.path.join(OUTPUT_DIR, "emoteflow_model.keras")
ONNX_SAVE_PATH = os.path.join(OUTPUT_DIR, "emoteflow_model.onnx")

# Hugging Face Hub settings
HF_REPO_ID = "charlykso/emoteflow-emotion-cnn"  # your HF repo
HF_COMMIT_MESSAGE = "Upload EmoteFlow emotion recognition model"

## Cell 4: Verify Dataset Paths

In [ ]:
# Verify and auto-detect dataset paths
import glob

# Auto-detect FER2013 train directory
possible_paths = glob.glob("/kaggle/input/**/train/angry", recursive=True)
if possible_paths:
    FER2013_TRAIN_DIR = os.path.dirname(possible_paths[0])  # parent of angry/
    FER2013_TEST_DIR = FER2013_TRAIN_DIR.replace("/train", "/test")
    print(f"Found FER2013 train dir: {FER2013_TRAIN_DIR}")
    print(f"Found FER2013 test dir:  {FER2013_TEST_DIR}")
    classes = sorted(os.listdir(FER2013_TRAIN_DIR))
    for cls in classes:
        count = len(os.listdir(os.path.join(FER2013_TRAIN_DIR, cls)))
        print(f"  {cls}: {count} images")
else:
    print("FER2013 dataset NOT FOUND anywhere under /kaggle/input/")
    print("→ Add dataset 'msambare/fer2013' via + Add Input")

# Auto-detect CK+ directory — look for any folder with emotion subfolders
CK_PLUS_DIR = None
for root, dirs, files in os.walk("/kaggle/input"):
    # Skip the FER2013 dataset we already found
    if "fer" in root.lower() and ("train" in root.lower() or "test" in root.lower()):
        continue
    # Check if this directory has emotion-class subfolders
    emotion_keywords = {"angry", "anger", "happy", "happiness", "sad", "sadness",
                        "surprise", "fear", "disgust", "contempt", "neutral"}
    subdirs_lower = {d.lower() for d in dirs}
    matches = subdirs_lower & emotion_keywords
    if len(matches) >= 3:  # at least 3 emotion folders found
        CK_PLUS_DIR = root
        break

if CK_PLUS_DIR:
    print(f"\nCK+ directory: FOUND at {CK_PLUS_DIR}")
    ck_classes = sorted(os.listdir(CK_PLUS_DIR))
    for cls in ck_classes:
        cls_path = os.path.join(CK_PLUS_DIR, cls)
        if os.path.isdir(cls_path):
            count = len(os.listdir(cls_path))
            print(f"  {cls}: {count} images")
else:
    print("\nCK+ directory: Not found. Fine-tuning will be skipped.")
    print("→ Add a CK+ dataset via + Add Input (search 'ckplus' or 'ck+48')")
    # Show what's actually in /kaggle/input for debugging
    print("\nCurrent inputs:")
    for root, dirs, files in os.walk("/kaggle/input"):
        level = root.replace("/kaggle/input", "").count(os.sep)
        if level < 3:
            indent = "  " * level
            print(f"{indent}{os.path.basename(root)}/")

## Cell 5: Data Loading with Augmentation

In [ ]:
# ─── Data Loading ────────────────────────────────────────────────────────────

train_datagen = ImageDataGenerator(
    rescale=1.0 / 255,
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    zoom_range=0.1,
    validation_split=0.1,
)

test_datagen = ImageDataGenerator(rescale=1.0 / 255)

train_gen = train_datagen.flow_from_directory(
    FER2013_TRAIN_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    color_mode="grayscale",
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    subset="training",
    shuffle=True,
)

val_gen = train_datagen.flow_from_directory(
    FER2013_TRAIN_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    color_mode="grayscale",
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    subset="validation",
    shuffle=False,
)

test_gen = test_datagen.flow_from_directory(
    FER2013_TEST_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    color_mode="grayscale",
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False,
)

print(f"\nTrain samples: {train_gen.samples}")
print(f"Validation samples: {val_gen.samples}")
print(f"Test samples: {test_gen.samples}")
print(f"Class indices: {train_gen.class_indices}")

## Cell 6: Preview Sample Images

In [ ]:
# Preview a batch of training images
images, labels = next(train_gen)
idx_to_label = {v: k for k, v in train_gen.class_indices.items()}

fig, axes = plt.subplots(2, 5, figsize=(15, 6))
for i, ax in enumerate(axes.flat):
    ax.imshow(images[i].squeeze(), cmap="gray")
    ax.set_title(idx_to_label[np.argmax(labels[i])])
    ax.axis("off")
plt.suptitle("Sample Training Images (with augmentation)", fontsize=14)
plt.tight_layout()
plt.show()

## Cell 7: Build CNN Model

In [ ]:
# ─── Model Architecture ─────────────────────────────────────────────────────
# 4 conv blocks (64→128→256→512) with BatchNorm + Dropout → Dense head

inputs = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 1))

# Block 1
x = layers.Conv2D(64, (3, 3), padding="same")(inputs)
x = layers.BatchNormalization()(x)
x = layers.Activation("relu")(x)
x = layers.Conv2D(64, (3, 3), padding="same")(x)
x = layers.BatchNormalization()(x)
x = layers.Activation("relu")(x)
x = layers.MaxPooling2D(pool_size=(2, 2))(x)
x = layers.Dropout(0.25)(x)

# Block 2
x = layers.Conv2D(128, (3, 3), padding="same")(x)
x = layers.BatchNormalization()(x)
x = layers.Activation("relu")(x)
x = layers.Conv2D(128, (3, 3), padding="same")(x)
x = layers.BatchNormalization()(x)
x = layers.Activation("relu")(x)
x = layers.MaxPooling2D(pool_size=(2, 2))(x)
x = layers.Dropout(0.25)(x)

# Block 3
x = layers.Conv2D(256, (3, 3), padding="same")(x)
x = layers.BatchNormalization()(x)
x = layers.Activation("relu")(x)
x = layers.Conv2D(256, (3, 3), padding="same")(x)
x = layers.BatchNormalization()(x)
x = layers.Activation("relu")(x)
x = layers.MaxPooling2D(pool_size=(2, 2))(x)
x = layers.Dropout(0.25)(x)

# Block 4
x = layers.Conv2D(512, (3, 3), padding="same")(x)
x = layers.BatchNormalization()(x)
x = layers.Activation("relu")(x)
x = layers.Conv2D(512, (3, 3), padding="same")(x)
x = layers.BatchNormalization()(x)
x = layers.Activation("relu")(x)
x = layers.MaxPooling2D(pool_size=(2, 2))(x)
x = layers.Dropout(0.25)(x)

# Classification head
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(512)(x)
x = layers.BatchNormalization()(x)
x = layers.Activation("relu")(x)
x = layers.Dropout(0.5)(x)
x = layers.Dense(256)(x)
x = layers.BatchNormalization()(x)
x = layers.Activation("relu")(x)
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(NUM_CLASSES, activation="softmax", name="emotion_output")(x)

model = keras.Model(inputs=inputs, outputs=outputs, name="EmoteFlow_CNN")
model.summary()

## Cell 8: Phase 1 — Train on FER2013

In [ ]:
# ─── Phase 1: Train on FER2013 ───────────────────────────────────────────────

# Compute class weights to handle FER2013 imbalance
labels = train_gen.classes
class_weights = dict(enumerate(
    compute_class_weight("balanced", classes=np.unique(labels), y=labels)
))
print(f"Class weights: {class_weights}")

# Compile
model.compile(
    optimizer=optimizers.Adam(learning_rate=LEARNING_RATE),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

# Callbacks
fer_callbacks = [
    callbacks.EarlyStopping(
        monitor="val_accuracy", patience=8, restore_best_weights=True, verbose=1
    ),
    callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=0.5, patience=3, min_lr=1e-7, verbose=1
    ),
    callbacks.ModelCheckpoint(
        MODEL_SAVE_PATH, monitor="val_accuracy", save_best_only=True, verbose=1
    ),
]

# Train
history_fer = model.fit(
    train_gen,
    epochs=EPOCHS_INITIAL,
    validation_data=val_gen,
    callbacks=fer_callbacks,
    class_weight=class_weights,
)

## Cell 9: FER2013 Training Curves

In [ ]:
# Plot FER2013 training history
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(history_fer.history["accuracy"], label="Train")
ax1.plot(history_fer.history["val_accuracy"], label="Validation")
ax1.set_title("FER2013 Training - Accuracy")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Accuracy")
ax1.legend()
ax1.grid(True)

ax2.plot(history_fer.history["loss"], label="Train")
ax2.plot(history_fer.history["val_loss"], label="Validation")
ax2.set_title("FER2013 Training - Loss")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Loss")
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "fer2013_training.png"), dpi=150)
plt.show()

## Cell 10: FER2013 Baseline Evaluation

In [ ]:
# ─── Evaluate on FER2013 test set ────────────────────────────────────────────

loss, accuracy = model.evaluate(test_gen, verbose=0)
print(f"Test Loss: {loss:.4f}")
print(f"Test Accuracy: {accuracy:.4f}")

predictions = model.predict(test_gen, verbose=0)
y_pred = np.argmax(predictions, axis=1)
y_true = test_gen.classes

idx_to_label = {v: k for k, v in test_gen.class_indices.items()}
target_names = [idx_to_label[i] for i in range(len(idx_to_label))]

print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=target_names))

# Confusion matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=target_names, yticklabels=target_names)
plt.title("EmoteFlow - Confusion Matrix (FER2013 Baseline)")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "confusion_matrix.png"), dpi=150)
plt.show()

## Cell 11: Phase 2 — Fine-tune on CK+ (Optional)

This cell only runs if you added a CK+ dataset. It freezes the first 2 conv blocks and trains with a lower learning rate.

In [ ]:
# ─── Phase 2: Fine-tune on CK+ ──────────────────────────────────────────────

if os.path.exists(CK_PLUS_DIR):
    print("CK+ dataset found. Starting fine-tuning...")

    # Freeze first 2 conv blocks
    freeze_until = 0
    pool_count = 0
    for i, layer in enumerate(model.layers):
        if isinstance(layer, layers.MaxPooling2D):
            pool_count += 1
            if pool_count == 2:
                freeze_until = i
                break

    for layer in model.layers[:freeze_until + 1]:
        layer.trainable = False

    trainable = sum(1 for l in model.layers if l.trainable)
    frozen = sum(1 for l in model.layers if not l.trainable)
    print(f"Frozen layers: {frozen}, Trainable layers: {trainable}")

    model.compile(
        optimizer=optimizers.Adam(learning_rate=FINETUNE_LR),
        loss="categorical_crossentropy",
        metrics=["accuracy"],
    )

    ck_datagen = ImageDataGenerator(
        rescale=1.0 / 255, rotation_range=10, horizontal_flip=True, validation_split=0.2
    )

    ck_train = ck_datagen.flow_from_directory(
        CK_PLUS_DIR, target_size=(IMG_SIZE, IMG_SIZE), color_mode="grayscale",
        batch_size=BATCH_SIZE, class_mode="categorical", subset="training", shuffle=True,
    )
    ck_val = ck_datagen.flow_from_directory(
        CK_PLUS_DIR, target_size=(IMG_SIZE, IMG_SIZE), color_mode="grayscale",
        batch_size=BATCH_SIZE, class_mode="categorical", subset="validation", shuffle=False,
    )

    ck_callbacks = [
        callbacks.EarlyStopping(monitor="val_accuracy", patience=5, restore_best_weights=True, verbose=1),
        callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-7, verbose=1),
        callbacks.ModelCheckpoint(MODEL_SAVE_PATH, monitor="val_accuracy", save_best_only=True, verbose=1),
    ]

    history_ck = model.fit(
        ck_train, epochs=EPOCHS_FINETUNE, validation_data=ck_val, callbacks=ck_callbacks
    )

    # Unfreeze all layers
    for layer in model.layers:
        layer.trainable = True

    # Plot CK+ training curves
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    ax1.plot(history_ck.history["accuracy"], label="Train")
    ax1.plot(history_ck.history["val_accuracy"], label="Validation")
    ax1.set_title("CK+ Fine-tuning - Accuracy")
    ax1.set_xlabel("Epoch"); ax1.set_ylabel("Accuracy"); ax1.legend(); ax1.grid(True)
    ax2.plot(history_ck.history["loss"], label="Train")
    ax2.plot(history_ck.history["val_loss"], label="Validation")
    ax2.set_title("CK+ Fine-tuning - Loss")
    ax2.set_xlabel("Epoch"); ax2.set_ylabel("Loss"); ax2.legend(); ax2.grid(True)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "ck+_fine-tuning.png"), dpi=150)
    plt.show()

    # Re-evaluate after fine-tuning
    loss, accuracy = model.evaluate(test_gen, verbose=0)
    print(f"\nPost Fine-tuning Test Accuracy: {accuracy:.4f}")

    predictions = model.predict(test_gen, verbose=0)
    y_pred = np.argmax(predictions, axis=1)
    print(classification_report(y_true, y_pred, target_names=target_names))

    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=target_names, yticklabels=target_names)
    plt.title("EmoteFlow - Confusion Matrix (After CK+ Fine-tuning)")
    plt.xlabel("Predicted"); plt.ylabel("Actual")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "confusion_matrix.png"), dpi=150)
    plt.show()
else:
    print(f"CK+ dataset not found at {CK_PLUS_DIR}. Skipping fine-tuning.")
    print("To add it: + Add Input → search for a CK+ emotion dataset")

## Cell 12: Save Keras Model

In [ ]:
# Save the final Keras model
# If kernel restarted after training, reload from the checkpoint saved by ModelCheckpoint
try:
    model
except NameError:
    if os.path.exists(MODEL_SAVE_PATH):
        print("model not in memory — loading from checkpoint...")
        model = keras.models.load_model(MODEL_SAVE_PATH)
    else:
        raise RuntimeError("No model in memory and no checkpoint found. Re-run cells 7 & 8 first.")

model.save(MODEL_SAVE_PATH)
print(f"Model saved to {MODEL_SAVE_PATH}")
print(f"Model size: {os.path.getsize(MODEL_SAVE_PATH) / (1024*1024):.1f} MB")

## Cell 13: Export to ONNX

In [ ]:
# ─── Export to ONNX for FastAPI serving ──────────────────────────────────────

import tf2onnx

spec = (tf.TensorSpec((None, IMG_SIZE, IMG_SIZE, 1), tf.float32, name="input"),)
model_proto, _ = tf2onnx.convert.from_keras(model, input_signature=spec,
                                              output_path=ONNX_SAVE_PATH)
print(f"ONNX model exported to {ONNX_SAVE_PATH}")
print(f"ONNX model size: {os.path.getsize(ONNX_SAVE_PATH) / (1024*1024):.1f} MB")

## Cell 14: Test ONNX Inference

In [ ]:
# Verify the ONNX model works
import onnxruntime as ort

session = ort.InferenceSession(ONNX_SAVE_PATH)
input_name = session.get_inputs()[0].name

# Test with a real sample from the test set
test_images, test_labels = next(test_gen)
sample = test_images[0:1]  # shape: (1, 48, 48, 1)

# Keras prediction
keras_pred = model.predict(sample, verbose=0)[0]

# ONNX prediction
onnx_pred = session.run(None, {input_name: sample.astype(np.float32)})[0][0]

print("Keras vs ONNX predictions (should be nearly identical):")
for label, k_p, o_p in zip(EMOTION_LABELS, keras_pred, onnx_pred):
    print(f"  {label:10s}  Keras: {k_p:.4f}  ONNX: {o_p:.4f}")

print(f"\nKeras  → {EMOTION_LABELS[np.argmax(keras_pred)]} ({np.max(keras_pred):.4f})")
print(f"ONNX   → {EMOTION_LABELS[np.argmax(onnx_pred)]} ({np.max(onnx_pred):.4f})")

## Cell 15: Upload to Hugging Face Hub

In [ ]:
# ─── Upload to Hugging Face Hub ──────────────────────────────────────────────

from huggingface_hub import HfApi, create_repo

hf_token = os.environ.get("HF_TOKEN")
if not hf_token:
    print("HF_TOKEN not set. Skipping upload.")
    print("Add it via: Add-ons → Secrets → HF_TOKEN")
else:
    api = HfApi(token=hf_token)

    # Create repo (no-op if it already exists)
    create_repo(HF_REPO_ID, repo_type="model", exist_ok=True, token=hf_token)

    # Generate model card
    accuracy_line = f"- **Test Accuracy**: {accuracy:.4f}"
    model_card = f"""---
tags:
  - emotion-recognition
  - cnn
  - fer2013
  - tensorflow
  - onnx
library_name: keras
pipeline_tag: image-classification
license: mit
---

# EmoteFlow Emotion Recognition CNN

A CNN model trained on FER2013 (and optionally fine-tuned on CK+) for real-time
student emotion recognition. Part of the EmoteFlow learning engagement platform.

## Labels

| Index | Emotion  |
|-------|----------|
| 0     | Angry    |
| 1     | Disgust  |
| 2     | Fear     |
| 3     | Happy    |
| 4     | Sad      |
| 5     | Surprise |
| 6     | Neutral  |

## Performance
{accuracy_line}
- **Input**: 48x48 grayscale image, normalized to [0, 1]
- **Output**: 7-class softmax probabilities

## Usage

```python
import numpy as np
import onnxruntime as ort

session = ort.InferenceSession("emoteflow_model.onnx")
image = np.random.rand(1, 48, 48, 1).astype(np.float32)  # your preprocessed frame
result = session.run(None, {{"input": image}})
emotion_probs = result[0][0]
```

## Training

- **Phase 1**: Trained on FER2013 ({EPOCHS_INITIAL} epochs max, early stopping)
- **Phase 2**: Fine-tuned on CK+ (if available, {EPOCHS_FINETUNE} epochs)
- **Architecture**: 4-block CNN (64→128→256→512) with BatchNorm + Dropout
"""

    model_card_path = os.path.join(OUTPUT_DIR, "README.md")
    with open(model_card_path, "w") as f:
        f.write(model_card)

    # Upload files
    files_to_upload = [
        (MODEL_SAVE_PATH, "emoteflow_model.keras"),
        (ONNX_SAVE_PATH, "emoteflow_model.onnx"),
        (model_card_path, "README.md"),
    ]

    confusion_path = os.path.join(OUTPUT_DIR, "confusion_matrix.png")
    if os.path.exists(confusion_path):
        files_to_upload.append((confusion_path, "confusion_matrix.png"))

    for local_path, repo_path in files_to_upload:
        api.upload_file(
            path_or_fileobj=local_path,
            path_in_repo=repo_path,
            repo_id=HF_REPO_ID,
            repo_type="model",
            commit_message=HF_COMMIT_MESSAGE,
            token=hf_token,
        )
        print(f"  Uploaded: {repo_path}")

    print(f"\nModel published to https://huggingface.co/{HF_REPO_ID}")

## Cell 16: Verify — Download from Hugging Face & Test

In [ ]:
# Verify the model can be downloaded and used from Hugging Face
from huggingface_hub import hf_hub_download

downloaded_path = hf_hub_download(
    repo_id=HF_REPO_ID,
    filename="emoteflow_model.onnx",
)
print(f"Downloaded model to: {downloaded_path}")

# Run inference with the downloaded model
hf_session = ort.InferenceSession(downloaded_path)
hf_input = hf_session.get_inputs()[0].name

dummy_image = np.random.rand(1, 48, 48, 1).astype(np.float32)
result = hf_session.run(None, {hf_input: dummy_image})[0][0]

predicted = EMOTION_LABELS[np.argmax(result)]
confidence = float(np.max(result))
print(f"\nTest prediction: {predicted} (confidence: {confidence:.4f})")
print("\nModel is ready for use in your FastAPI backend via emotion_predictor.py")

---

## Done!

Your model is now on Hugging Face at `charlykso/emoteflow-emotion-cnn`.

In your FastAPI backend, use `emotion_predictor.py` to load it:

```python
from emotion_predictor import EmotionPredictor

predictor = EmotionPredictor()  # auto-downloads from HF
result = predictor.predict(face_image)
# → {"emotion": "Happy", "confidence": 0.92, "scores": {...}}
```